# RQ2-v3: geometry → gradient second moment → sampler variance

CPU-only post-processing of the completed cross-subnet interaction probe. This notebook does not load a model/checkpoint, run forward/backward, use accuracy/test data, or train anything. It measures `m_i = E||g_i||²`, tests whether frozen functional mass predicts it beyond FLOPs, and compares the empirical variance of unbiased importance-corrected gradient estimators.

In [ ]:
import os, subprocess, sys, json, shutil, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Missing Kaggle secret github_token'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN_RUNTIME':github_token})
try:
    command = ['git','-C',str(PROJECT_ROOT),'pull','--ff-only'] if (PROJECT_ROOT/'.git').is_dir() else ['git','clone','https://github.com/duyh80456-code/new-pruning.git',str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True); github_token = None
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT))
GIT_COMMIT = subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('Commit:', GIT_COMMIT)
print('CPU-only notebook; Kaggle accelerator should be None')

## Locate the completed interaction-probe output

In [ ]:
import importlib, rq2_gradient_variance_v3
rq2_gradient_variance_v3 = importlib.reload(rq2_gradient_variance_v3)
INPUT_ROOT = Path('/kaggle/input')
INTERACTION_ROOT = rq2_gradient_variance_v3.find_interaction_probe_root(
    INPUT_ROOT, '/kaggle/working/materialized-rq2-interaction-probe'
)
print('Interaction root:', INTERACTION_ROOT)
print(json.dumps(json.loads((INTERACTION_ROOT/'metadata.json').read_text()), indent=2))

## Run the frozen v3 diagnostic
The estimand is the uniformly weighted mean gradient over the 14 interior widths, so `w_i=1/14`. `Uniform-dynamic` means equal inclusion probability over all 14 widths. The historical fixed `{.50,.75}` support is also reported, but its unbiased importance estimator is infeasible because twelve inclusion probabilities are zero.

In [ ]:
OUTPUT_DIR = Path('/kaggle/working/rq2-gradient-variance-v3')
summary = rq2_gradient_variance_v3.run_gradient_variance_v3(INTERACTION_ROOT, OUTPUT_DIR)
summary['git_commit'] = GIT_COMMIT
(OUTPUT_DIR/'rq2_v3_diagnostic_summary.json').write_text(json.dumps(summary, indent=2)+'\n')
print(json.dumps(summary, indent=2))

## Inspect the bridge and variance comparison

In [ ]:
import pandas as pd
from IPython.display import display, Image
moments = pd.read_csv(OUTPUT_DIR/'gradient_second_moment_by_width.csv')
correlations = pd.read_csv(OUTPUT_DIR/'gradient_second_moment_correlations.csv')
regression = pd.read_csv(OUTPUT_DIR/'gradient_second_moment_regression.csv')
variance = pd.read_csv(OUTPUT_DIR/'importance_corrected_estimator_variance.csv')
display(moments)
display(correlations)
display(regression)
display(variance)
display(Image(filename=str(OUTPUT_DIR/'geometry_flops_vs_gradient_moment.png')))
display(Image(filename=str(OUTPUT_DIR/'importance_corrected_variance.png')))

## Validate and export

In [ ]:
required = [
    'gradient_second_moment_by_width.csv',
    'gradient_second_moment_correlations.csv',
    'gradient_second_moment_regression.csv',
    'gradient_second_moment_loow_predictions.csv',
    'importance_corrected_estimator_variance.csv',
    'geometry_flops_vs_gradient_moment.png',
    'importance_corrected_variance.png',
    'rq2_v3_diagnostic_summary.json',
]
missing = [name for name in required if not (OUTPUT_DIR/name).is_file() or (OUTPUT_DIR/name).stat().st_size == 0]
assert not missing, f'Missing v3 artifacts: {missing}'
saved = json.loads((OUTPUT_DIR/'rq2_v3_diagnostic_summary.json').read_text())
assert saved['training_performed'] is False
assert saved['model_or_checkpoint_loaded'] is False
assert saved['gpu_used'] is False and saved['test_used'] is False
bundle_path = Path('/kaggle/working/rq2-gradient-variance-v3.zip')
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in OUTPUT_DIR.rglob('*'):
        if path.is_file(): bundle.write(path, path.relative_to(OUTPUT_DIR))
print('Download/persist:', bundle_path, f'{bundle_path.stat().st_size/2**20:.1f} MiB')
bundle_path